In [ ]:
#Tool Creation
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import ArxivAPIWrapper,WikipediaAPIWrapper

api_wrapper_wiki=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki)

api_wrapper_arxiv=ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=250)
arxiv=ArxivQueryRun(api_wrapper=api_wrapper_arxiv) 

tools=[wiki,arxiv]


In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS 
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader=WebBaseLoader("https://docs.langchain.com/oss/python/langchain/overview")
docs=loader.load()
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=100)
documents=text_splitter.split_documents(docs)
vectordb=FAISS.from_documents(documents,OpenAIEmbeddings())
retriever=vectordb.as_retriever()




In [ ]:
from langchain_core.tools import create_retriever_tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

llm = ChatOpenAI(model="gpt-4o-mini")

# Your retriever (example)
# retriever = ... (your retriever instance)

retriever_tool = create_retriever_tool(
    retriever,
    "langsmith_search",                           # name (no spaces)
    "Search LangSmith documentation. Use this for any LangSmith questions."
)

tools = [wiki,arxiv,retriever_tool]

agent = create_agent(
    llm,
    tools,
    system_prompt="You are a helpful assistant with LangSmith knowledge."
)

result = agent.invoke({  
    "messages": [{"role": "user", "content": "Explain transformer"}]
})
print(retriever_tool.name)         # "langsmith_search"
print(retriever_tool.description)  # "Search LangSmith documentation..."
print(result)

In [ ]:
import streamlit as st
from langchain_groq import ChatGroq
from langchain_community.utilities import ArxivAPIWrapper,WikipediaAPIWrapper
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun,DuckDuckGoSearchRun
from langchain.agents import initialize_agent,AgentType

In [ ]:
!pip install -U ddgs